In [0]:

# 1. Load Fragrantica perfumes from the Delta table.

df_fragrantica = spark.table(
    "main.fragrance_analytics.fragrantica_perfumes"
)

print("Fragrantica perfumes:", df_fragrantica.count())
df_fragrantica.printSchema()


# 2. Load Scento source files.


# 2. Load Scento datasets.

import requests
import pandas as pd
from io import StringIO

SCENTO_BASE = (
    "https://huggingface.co/datasets/"
    "scento/perfume-database/resolve/main"
)

SCENTO_API = "https://www.scento.com/api/perfume/v1"
TENML_API = "https://mcp.10ml.com/api/v1"

HEADERS = {
    "User-Agent": "FragranceRecommendationPlatform/0.1"
}

SOURCE_FILES = {
    "perfumes": f"{SCENTO_BASE}/perfumes.csv",
    "houses": f"{SCENTO_BASE}/houses.csv",
    "materials": f"{SCENTO_BASE}/materials.csv"
}

scento_data = {}

for name, url in SOURCE_FILES.items():
    response = requests.get(
        url,
        headers=HEADERS,
        timeout=30
    )
    response.raise_for_status()

    df = pd.read_csv(StringIO(response.text))
    scento_data[name] = df

    print(f"\n{name.upper()}")
    print(f"Rows: {len(df):,}")
    print(f"Columns: {len(df.columns)}")
    print(df.dtypes)


# 3. Verify loaded datasets.


# 3. Verify both datasets.

print("FRAGRANTICA")
print("Rows:", df_fragrantica.count())
print("Distinct brands:", df_fragrantica.select("brand_name").distinct().count())

print("\nSCENTO")
print("Perfumes:", len(scento_data["perfumes"]))
print("Houses:", len(scento_data["houses"]))
print("Materials:", len(scento_data["materials"]))
print("Distinct houses:", scento_data["perfumes"]["house"].nunique())


In [0]:

# 1. Define candidate brands for portfolio analysis.

brand_groups = {
    "Luxury": [
        "Dior",
        "Guerlain",
        "Chanel",
        "Hermes",
        "Mugler",
        "Yves Saint Laurent",
        "Tom Ford"
    ],
    "Niche": [
        "Nicolai Parfumeur Createur",
        "Nasomatto",
        "Memo Paris",
        "Byredo",
        "27 87",
        "Maison Francis Kurkdjian",
        "Serge Lutens"
    ]
}

brand_candidates = [
    (brand, group)
    for group, brands in brand_groups.items()
    for brand in brands
]

print(f"Candidate brands: {len(brand_candidates)}")

for group, brands in brand_groups.items():
    print(f"\n{group}: {len(brands)}")
    for brand in brands:
        print(f"  - {brand}")


In [0]:

# 2. Check candidate brand coverage in Fragrantica and Scento.

import pandas as pd
from pyspark.sql import functions as F

# Candidate brand reference.
candidate_df = pd.DataFrame(
    brand_candidates,
    columns=["brand_name", "brand_group"]
)

candidate_df["brand_key"] = (
    candidate_df["brand_name"].str.strip().str.casefold()
)

# Fragrantica counts.
df_fragrantica = spark.table(
    "main.fragrance_analytics.fragrantica_perfumes"
)

fragrantica_counts = (
    df_fragrantica
    .filter(F.col("brand_name").isNotNull())
    .withColumn(
        "brand_key",
        F.lower(F.trim(F.col("brand_name")))
    )
    .groupBy("brand_key")
    .agg(
        F.countDistinct("id").alias("fragrantica_count")
    )
    .toPandas()
)

# Scento counts.
df_scento = scento_data["perfumes"].copy()

df_scento["brand_key"] = (
    df_scento["house"]
    .astype("string")
    .str.strip()
    .str.casefold()
)

scento_counts = (
    df_scento
    .dropna(subset=["brand_key"])
    .groupby("brand_key")["id"]
    .nunique()
    .reset_index(name="scento_count")
)

# Combine coverage results.
brand_coverage = (
    candidate_df
    .merge(fragrantica_counts, on="brand_key", how="left")
    .merge(scento_counts, on="brand_key", how="left")
)

brand_coverage["fragrantica_count"] = (
    brand_coverage["fragrantica_count"].fillna(0).astype(int)
)

brand_coverage["scento_count"] = (
    brand_coverage["scento_count"].fillna(0).astype(int)
)

brand_coverage["both_datasets"] = (
    (brand_coverage["fragrantica_count"] > 0) &
    (brand_coverage["scento_count"] > 0)
)

brand_coverage["combined_source_records"] = (
    brand_coverage["fragrantica_count"] +
    brand_coverage["scento_count"]
)

brand_coverage = brand_coverage.sort_values(
    ["both_datasets", "combined_source_records"],
    ascending=[False, False]
).reset_index(drop=True)

display(
    brand_coverage[
        [
            "brand_name",
            "brand_group",
            "fragrantica_count",
            "scento_count",
            "both_datasets",
            "combined_source_records"
        ]
    ]
)


In [0]:

# 5. Inspect alternative brand and house names.

search_terms = [
    "chanel",
    "herm",
    "nicolai",
    "2787",
    "27 87",
    "serge lutens",
    "nasomatto"
]

print("FRAGRANTICA BRAND NAMES")

fragrantica_brands = (
    df_fragrantica
    .select("brand_name")
    .distinct()
    .filter(F.col("brand_name").isNotNull())
)

for term in search_terms:
    print(f"\nSearch: {term}")
    display(
        fragrantica_brands
        .filter(F.lower(F.col("brand_name")).contains(term))
        .orderBy("brand_name")
    )

print("\nSCENTO HOUSE NAMES")

scento_houses = sorted(
    scento_data["houses"]["name"].dropna().unique()
)

for term in search_terms:
    matches = [
        house for house in scento_houses
        if term in house.casefold()
    ]
    print(f"\nSearch: {term}")
    print(matches)


In [0]:

# Normalize brand names for matching across datasets.

import unicodedata

def normalize_brand(name):
    if pd.isna(name):
        return None

    name = unicodedata.normalize("NFKD", str(name))
    name = "".join(
        char for char in name
        if not unicodedata.combining(char)
    )

    return name.strip().casefold()
